# B2.11 · Evaluating a security harness

**Function B — Application Security with an AI SDLC → The Harness that Runs the SDLC**  ·  *AI for Security*

Builds on **[B2.10 · Choosing the model backbone](https://spbreed.github.io/cyber-commons/lessons/B2.10.html)**.

| | |
|---|---|
| Open-source tooling | Cyber Commons eval harness, Checkov, CyberGym, Inspect |
| Open-weight models | Llama 3.3, GLM-4.6, Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

A single successful run hides how unreliable an agent is, and a hallucinated finding is indistinguishable from a real one until something checks. Evaluation is the only part of this chapter that tells you whether the rest of it worked.

## 2 · The framework

```
   corpus with known answers
        |
        v
   harness run --> findings --> scored against the key
                                     |
                    +----------------+----------------+
                    | recall  precision  cost  time   |
                    +---------------------------------+

   a hallucinated finding looks exactly like a real one
   until something with the answers checks
```

This is the flagship lesson of the track: how to tell whether a security harness
is any good.

Four stages, in order, each answering a different question:

1. **Ingest** — did the output parse and match the schema?
2. **Path matching** — is the finding about the file we asked about?
3. **Expert proxy** — is it right? Scored {0, 0.5, 1}.
4. **Dual judges** — is the reasoning sound? Two judges, aggregated by **MIN**.

And the distinction the whole lesson exists for:

> **Conformance** is schema validity. With structured output it is ~100% by
> construction. It is a build-health signal.
>
> **Accuracy** is correctness. It is the number that means something.

Quoting conformance as quality — "our harness scores 100%" — is the single most
common way a security evaluation misleads its own sponsors.

There is also one implementation detail that silently randomises everyone's
results, and it is a single line: **file matching must use parent directory plus
filename, never the bare basename.** Public corpora reuse `1.py` and `3.c` across
every CWE directory.

## 3 · Stage 1 — ingest, where conformance is decided

In [ ]:
import json
from dataclasses import dataclass, field

@dataclass
class Answer:
    qid: str; cwe: str = ""; file: str = ""; line: int = 0; rationale: str = ""
    REQUIRED = ("qid", "cwe", "file", "rationale")

    @classmethod
    def parse(cls, raw):
        try:
            d = json.loads(raw)
        except json.JSONDecodeError as e:
            return None, f"non-conforming: not JSON ({e.msg})"
        missing = [k for k in cls.REQUIRED if not d.get(k)]
        if missing:
            return None, f"non-conforming: missing {missing}"
        return cls(str(d["qid"]), str(d["cwe"]).upper(), str(d["file"]),
                   int(d.get("line", 0)), str(d["rationale"])), "conforming"

@dataclass
class Truth:
    qid: str; cwe: str; file: str; line: int = 0

TRUTHS = {
 "q1": Truth("q1", "CWE-89",  "CWE-89/1.py"),
 "q2": Truth("q2", "CWE-78",  "CWE-78/1.py"),
 "q3": Truth("q3", "CWE-22",  "CWE-22/3.c"),
 "q4": Truth("q4", "CWE-798", "CWE-798/2.py"),
}
ANSWERS = {
 "q1": '{"qid":"q1","cwe":"CWE-89","file":"CWE-89/1.py","line":2,'
       '"rationale":"user input is concatenated into the query string"}',
 "q2": '{"qid":"q2","cwe":"CWE-89","file":"CWE-78/1.py","line":3,'
       '"rationale":"untrusted input reaches a shell"}',
 "q3": '{"qid":"q3","cwe":"CWE-22","file":"CWE-89/1.py","line":1,'
       '"rationale":"path built from user input"}',
 "q4": 'I think this file contains a hardcoded credential.',
}
for qid, raw in ANSWERS.items():
    _, note = Answer.parse(raw)
    print(f"{qid}: {note}")

## 4 · Stage 2 — the one line that decides whether this is a benchmark

Public corpora reuse filenames across directories. Match on the basename and you score an answer about CWE-79 against the ground truth for CWE-89 — and your accuracy becomes a random variable.

In [ ]:
def path_key(path):
    """Parent directory + filename. NEVER the bare basename."""
    parts = [p for p in path.replace("\\", "/").split("/") if p not in ("", ".")]
    return "/".join(parts[-2:]) if len(parts) > 1 else (parts[-1] if parts else "")

def basename(path):
    return path.replace("\\", "/").split("/")[-1]

pairs = [("CWE-89/1.py", "CWE-79/1.py"), ("a/CWE-22/3.c", "b/CWE-78/3.c")]
print(f"{'pair':34s}{'basename match':17s}path_key match")
print("-" * 66)
for a, b in pairs:
    print(f"{a + '  vs  ' + b:34s}{str(basename(a)==basename(b)):17s}"
          f"{path_key(a)==path_key(b)}")
print("\nq3 answered about CWE-89/1.py when the truth is CWE-22/3.c.")
print(f"   basename says match: {basename('CWE-89/1.py') == basename('CWE-22/3.c')}")
print(f"   path_key says match: {path_key('CWE-89/1.py') == path_key('CWE-22/3.c')}")

## 5 · Stages 3 and 4 — expert proxy and two judges

Half credit is not politeness. "Right file, wrong vulnerability class" is a genuinely different failure from "wrong file entirely", and averaging them away hides which one your harness is making.

Two judges aggregated by **MIN**, not mean — judges exist to catch each other, and averaging lets the lenient one carry the strict one's failures.

In [ ]:
def path_match(a, t): return path_key(a.file) == path_key(t.file)
def cwe_match(a, t):  return a.cwe == t.cwe.upper()

def expert_proxy(a, t):
    if not path_match(a, t): return 0.0
    return 1.0 if cwe_match(a, t) else 0.5

MECHANISM = ("concatenat", "unsanitis", "unsanitiz", "untrusted", "user input",
             "interpolat", "taint", "unvalidated")
def judge_strict(a, t):
    if expert_proxy(a, t) < 1.0: return 0.0
    return 1.0 if any(w in a.rationale.lower() for w in MECHANISM) else 0.5
def judge_lenient(a, t):
    return 1.0 if cwe_match(a, t) else 0.0

@dataclass
class Report:
    total: int = 0; conforming: int = 0
    expert_sum: float = 0.0; judge_sum: float = 0.0
    failures: list = field(default_factory=list)
    @property
    def conformance(self): return self.conforming / self.total if self.total else 0
    @property
    def expert_accuracy(self): return self.expert_sum / self.total if self.total else 0
    @property
    def judge_accuracy(self): return self.judge_sum / self.total if self.total else 0
    def render(self):
        return (f"  questions            {self.total}\n"
                f"  conformance          {self.conformance:.4f}   "
                f"← schema validity. Structural. NOT quality.\n"
                f"  expert accuracy      {self.expert_accuracy:.4f}   ← correctness\n"
                f"  judge accuracy (MIN) {self.judge_accuracy:.4f}\n"
                f"  failures             {len(self.failures)}")

def evaluate(answers, truths):
    rep = Report(total=len(truths))
    for qid, t in truths.items():
        a, note = Answer.parse(answers.get(qid, ""))
        if a is None:
            rep.failures.append((qid, "1-ingest", note)); continue
        rep.conforming += 1
        e = expert_proxy(a, t)
        j = min(judge_strict(a, t), judge_lenient(a, t))
        rep.expert_sum += e; rep.judge_sum += j
        if e < 1.0:
            why = ("wrong file" if not path_match(a, t)
                   else f"right file, wrong class (said {a.cwe}, truth {t.cwe})")
            rep.failures.append((qid, "3-expert", why))
    return rep

rep = evaluate(ANSWERS, TRUTHS)
print(rep.render())
print("\nfailures:")
for qid, stage, why in rep.failures:
    print(f"   {qid}  [{stage}]  {why}")

## 6 · The control — never report one number

Here is what happens when the harness is upgraded to emit structured output. Conformance goes to 1.0. Nothing about its capability changed.

In [ ]:
STRUCTURED = dict(ANSWERS)
STRUCTURED["q4"] = ('{"qid":"q4","cwe":"CWE-798","file":"CWE-798/2.py","line":1,'
                    '"rationale":"a credential is hardcoded"}')
rep2 = evaluate(STRUCTURED, TRUTHS)
print("after adding structured output:")
print(rep2.render())
print(f"\nconformance  {rep.conformance:.2f} → {rep2.conformance:.2f}   "
      f"(+{rep2.conformance-rep.conformance:.2f})")
print(f"expert acc   {rep.expert_accuracy:.2f} → {rep2.expert_accuracy:.2f}   "
      f"(+{rep2.expert_accuracy-rep.expert_accuracy:.2f})")
print("\nA press release could truthfully say 'conformance improved to 100%'.")
print("The harness still gets half the questions wrong.")

def gameable(answers):
    parsed = [Answer.parse(r)[0] for r in answers.values()]
    ok = [p for p in parsed if p]
    cwes = [p.cwe for p in ok]
    maj = max(set(cwes), key=cwes.count) if cwes else ""
    return {"conformance": round(len(ok)/len(answers), 3),
            "majority_class": maj,
            "accuracy_by_always_guessing_majority":
                round(cwes.count(maj)/len(cwes), 3) if cwes else 0}
print("\nwithout any capability at all:", gameable(STRUCTURED))
assert rep2.conformance == 1.0 and rep2.expert_accuracy < 0.7

## What you just proved

q1–q3 conform and q4 does not, giving conformance 0.75 against expert accuracy 0.375 — q1 scores 1.0, q2 scores 0.5 (right file, wrong class), q3 and q4 score 0. The basename comparison wrongly matches `CWE-89/1.py` with `CWE-79/1.py` where `path_key` does not. Adding structured output raises conformance to 1.00 while expert accuracy moves to 0.625, and the gameability check shows high conformance with no capability.

## Your turn

Take an eval number your organisation has quoted — internally or externally — and determine which of the two it was. Then check the file matcher in whatever produced it. Both checks take an hour and one of them usually changes the number.

---

**Next → [B2.12 · Reliability and cost under non-determinism](https://spbreed.github.io/cyber-commons/lessons/B2.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*